# Breast Cancer Detection - EfficientNet-B4 V13 Optimized

## CBIS-DDSM Mammography Classification Pipeline

**Version:** V13 - Optimized Fine-Tuning Based on V12 Analysis

**Architecture:** EfficientNet-B4 with Conservative Transfer Learning

---

### Key Improvements from V12 Analysis:

1. **Increased Regularization**: L2: 1e-4 → 3e-4, Dropout: 0.4 → 0.5
2. **Optimized Training Schedule**: Stage 2 focus, reduced Stage 3
3. **Conservative Unfreezing**: Freeze more layers in each stage
4. **Lower Learning Rates**: Reduced LR for fine-tuning stages
5. **Cosine Annealing**: Smoother LR decay
6. **Enhanced Augmentation**: More aggressive augmentation
7. **Smarter Early Stopping**: Stop sooner on plateau

### V12 Baseline (to beat):
- Best individual model: **0.7955 AUC** (Model 2, Stage 2, epoch 46)
- Ensemble result: 0.770 AUC

## 1. Environment Setup

In [ ]:
import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNTIME_ENV = "colab"
except ImportError:
    RUNTIME_ENV = "local"

if RUNTIME_ENV == "colab":
    import subprocess
    subprocess.run(['pip', 'install', 'pydicom', '-q'], capture_output=True)

import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

import cv2
import pydicom
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback, LearningRateScheduler
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

physical_gpus = tf.config.list_physical_devices('GPU')
GPU_AVAILABLE = len(physical_gpus) > 0

def get_gpu_memory_gb():
    if not GPU_AVAILABLE:
        return 0
    try:
        gpu_details = tf.config.experimental.get_device_details(physical_gpus[0])
        gpu_name = gpu_details.get('device_name', '').lower()
        if 'h100' in gpu_name:
            return 80
        elif 'a100' in gpu_name:
            return 40
        elif 'l4' in gpu_name:
            return 24
        elif 't4' in gpu_name:
            return 15
        elif 'v100' in gpu_name:
            return 16
        else:
            if '80gb' in gpu_name or '80g' in gpu_name:
                return 80
            elif '40gb' in gpu_name or '40g' in gpu_name:
                return 40
            return 15
    except Exception:
        return 15

GPU_MEMORY_GB = get_gpu_memory_gb()

if GPU_AVAILABLE:
    for gpu in physical_gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    GPU_NAME = tf.config.experimental.get_device_details(physical_gpus[0]).get('device_name', 'Unknown')
else:
    GPU_NAME = "None"

def print_memory_usage(stage=""):
    if RUNTIME_ENV == "colab":
        try:
            import psutil
            mem = psutil.virtual_memory()
            print(f"[{stage}] RAM: {mem.used/1e9:.1f}GB / {mem.total/1e9:.1f}GB ({mem.percent}%)")
        except ImportError:
            pass
    if GPU_AVAILABLE:
        try:
            gpu_mem = tf.config.experimental.get_memory_info('GPU:0')
            if gpu_mem:
                print(f"[{stage}] GPU: {gpu_mem.get('current', 0)/1e9:.2f}GB")
        except Exception:
            pass

print("-" * 60)
print("V13 OPTIMIZED - ENVIRONMENT CONFIGURATION")
print("-" * 60)
print(f"Runtime:        {RUNTIME_ENV.upper()}")
print(f"TensorFlow:     {tf.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"GPU Available:  {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print(f"GPU Name:       {GPU_NAME}")
    print(f"GPU Memory:     {GPU_MEMORY_GB} GB (detected)")
print("-" * 60)

## 2. Optimized Configuration

In [ ]:
class ExperimentConfigV13:
    def __init__(self, gpu_memory_gb=15):
        self.gpu_memory_gb = gpu_memory_gb
        self.runtime_env = RUNTIME_ENV
        self.version = "V13_Optimized"
        
        if self.runtime_env == "colab":
            self.base_path = Path('/content/drive/MyDrive/CBIS-DDSM-data')
            self.dicom_path = self.base_path / 'CBIS-DDSM'
            self.csv_path = self.base_path / 'csv'
            self.cache_path = self.base_path / 'preprocessed_cache_roi_efficientnet'
            self.checkpoint_path = self.base_path / 'checkpoints_v13_optimized'
            self.results_path = self.base_path / 'results_v13_optimized'
        else:
            self.base_path = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
            self.dicom_path = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
            self.csv_path = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
            self.cache_path = self.base_path / 'preprocessed_cache_roi_efficientnet'
            self.checkpoint_path = self.base_path / 'checkpoints_v13_optimized'
            self.results_path = self.base_path / 'results_v13_optimized'
        
        self.calc_train_csv = self.csv_path / 'calc_case_description_train_set.csv'
        self.calc_test_csv = self.csv_path / 'calc_case_description_test_set.csv'
        self.mass_train_csv = self.csv_path / 'mass_case_description_train_set.csv'
        self.mass_test_csv = self.csv_path / 'mass_case_description_test_set.csv'
        
        self.image_size = (380, 380)
        self.image_channels = 3
        self.use_clahe = True
        self.clahe_clip_limit = 2.0
        self.clahe_tile_size = (8, 8)
        
        # GPU-ADAPTIVE CONFIGURATION
        if gpu_memory_gb >= 40:
            self.batch_size = 32
            self.batch_size_finetune = 16
            self.batch_size_inference = 64
            self.dense_units = 1024
            self.ensemble_size = 3
            self.use_mixed_precision = True
            self.tta_augmentations = 8
            self.shuffle_buffer = 2000
        elif gpu_memory_gb >= 20:
            self.batch_size = 16
            self.batch_size_finetune = 8
            self.batch_size_inference = 32
            self.dense_units = 1024
            self.ensemble_size = 3
            self.use_mixed_precision = True
            self.tta_augmentations = 8
            self.shuffle_buffer = 1000
        else:
            self.batch_size = 8
            self.batch_size_finetune = 4
            self.batch_size_inference = 16
            self.dense_units = 512
            self.ensemble_size = 1
            self.use_mixed_precision = True
            self.tta_augmentations = 4
            self.shuffle_buffer = 500
        
        self.ensemble_seeds = [42, 123, 456][:self.ensemble_size]
        
        # V13 OPTIMIZED TRAINING SCHEDULE
        self.stage1_epochs = 25
        self.stage1_lr = 3e-4
        self.stage2_epochs = 50
        self.stage2_lr = 5e-5
        self.stage2_frozen_layers = 350
        self.stage3_epochs = 25
        self.stage3_lr = 1e-5
        self.stage3_frozen_layers = 200
        
        self.use_lr_warmup = True
        self.warmup_epochs = 5
        self.warmup_start_lr = 1e-7
        self.use_cosine_annealing = True
        
        self.train_split = 0.70
        self.val_split = 0.15
        self.test_split = 0.15
        
        # V13 STRONGER REGULARIZATION
        self.l1_regularization = 0.0
        self.l2_regularization = 3e-4
        self.dropout_rate = 0.5
        self.label_smoothing = 0.15
        self.gradient_clip_norm = 1.0
        self.use_spatial_dropout = True
        self.spatial_dropout_rate = 0.2
        
        self.use_class_weight = True
        self.malignant_weight_multiplier = 2.5
        self.class_weights = None
        
        self.use_focal_loss = True
        self.focal_gamma = 2.0
        self.focal_alpha = 0.7
        
        # V13 IMPROVED CALLBACKS
        self.early_stop_patience = 15
        self.lr_reduce_patience = 7
        self.lr_reduce_factor = 0.5
        self.min_epochs_stage1 = 15
        self.min_epochs_stage2 = 25
        self.min_epochs_stage3 = 15
        
        self.use_tta = True
        self.prefetch_buffer = tf.data.AUTOTUNE
        self.num_parallel_calls = tf.data.AUTOTUNE
        
        self._create_directories()
    
    def _create_directories(self):
        for path in [self.cache_path, self.checkpoint_path, self.results_path]:
            path.mkdir(parents=True, exist_ok=True)
    
    def display(self):
        total_epochs = (self.stage1_epochs + self.stage2_epochs + self.stage3_epochs)
        total_training_epochs = total_epochs * self.ensemble_size
        print("-" * 60)
        print(f"V13 OPTIMIZED CONFIGURATION")
        print("-" * 60)
        print(f"GPU Memory:           {self.gpu_memory_gb} GB")
        print(f"Mixed Precision:      {self.use_mixed_precision}")
        print("-" * 60)
        print("Model Architecture:")
        print(f"  Base Model:         EfficientNet-B4")
        print(f"  Input Size:         {self.image_size[0]}x{self.image_size[1]}")
        print(f"  Dense Units:        {self.dense_units}")
        print(f"  Dropout Rate:       {self.dropout_rate}")
        print(f"  L2 Regularization:  {self.l2_regularization}")
        print(f"  Spatial Dropout:    {self.spatial_dropout_rate}")
        print("-" * 60)
        print("Training Configuration:")
        print(f"  Batch Size:         {self.batch_size}")
        print(f"  Ensemble Models:    {self.ensemble_size}")
        print(f"  Stage 1:            {self.stage1_epochs} epochs @ LR={self.stage1_lr}")
        print(f"  Stage 2:            {self.stage2_epochs} epochs @ LR={self.stage2_lr} (FOCUS)")
        print(f"  Stage 3:            {self.stage3_epochs} epochs @ LR={self.stage3_lr} (reduced)")
        print(f"  Total Epochs:       {total_training_epochs}")
        print(f"  Cosine Annealing:   {self.use_cosine_annealing}")
        print("-" * 60)
        print("Unfreezing Strategy:")
        print(f"  Stage 2 frozen:     {self.stage2_frozen_layers} layers")
        print(f"  Stage 3 frozen:     {self.stage3_frozen_layers} layers")
        print("-" * 60)
        print("Early Stopping:")
        print(f"  Patience:           {self.early_stop_patience}")
        print("-" * 60)

config = ExperimentConfigV13(gpu_memory_gb=GPU_MEMORY_GB)
config.display()

if GPU_AVAILABLE and config.use_mixed_precision:
    try:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        print("Mixed precision (FP16) enabled")
    except Exception as e:
        config.use_mixed_precision = False
        print(f"Mixed precision not available: {e}")

## 3. ROI Data Preprocessing

In [ ]:
def apply_clahe(image, clip_limit=2.0, tile_size=(8, 8)):
    if image.dtype != np.uint8:
        image = ((image - image.min()) / (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    enhanced = clahe.apply(image)
    return enhanced.astype(np.float32)

def find_dicom_file(case_dir_name, base_path):
    case_path = base_path / case_dir_name
    if not case_path.exists():
        return None
    try:
        for date_folder in case_path.iterdir():
            if date_folder.is_dir():
                for series_folder in date_folder.iterdir():
                    if series_folder.is_dir():
                        for dcm_file in series_folder.iterdir():
                            if dcm_file.suffix == '.dcm':
                                return dcm_file
    except Exception:
        pass
    return None

def load_dicom_image(dicom_path, target_size=(380, 380), use_clahe=True):
    try:
        dcm = pydicom.dcmread(str(dicom_path))
        image = dcm.pixel_array.astype(np.float32)
        if hasattr(dcm, 'PhotometricInterpretation'):
            if dcm.PhotometricInterpretation == 'MONOCHROME1':
                image = image.max() - image
        image = ((image - image.min()) / (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)
        if use_clahe:
            image = apply_clahe(image)
        else:
            image = image.astype(np.float32)
        image = cv2.resize(image, target_size, interpolation=cv2.INTER_LANCZOS4)
        image = np.stack([image] * 3, axis=-1)
        return image
    except Exception:
        return None

def preprocess_roi_dataset(config, csv_files, split_name='train'):
    images = []
    labels = []
    processed_cases = set()
    for csv_file in csv_files:
        if not csv_file.exists():
            continue
        df = pd.read_csv(csv_file)
        path_col = None
        for col_name in ['cropped image file path', 'ROI mask file path', 'image file path']:
            if col_name in df.columns:
                path_col = col_name
                break
        if path_col is None:
            continue
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {csv_file.name}"):
            pathology = str(row.get('pathology', '')).upper()
            if 'MALIGNANT' in pathology:
                label = 1
            elif 'BENIGN' in pathology:
                label = 0
            else:
                continue
            rel_path = row.get(path_col, None)
            if pd.isna(rel_path) or rel_path is None:
                continue
            case_dir_name = Path(rel_path).parts[0]
            case_key = (case_dir_name, label)
            if case_key in processed_cases:
                continue
            processed_cases.add(case_key)
            dicom_file = find_dicom_file(case_dir_name, config.dicom_path)
            if dicom_file is None:
                continue
            image = load_dicom_image(dicom_file, target_size=config.image_size, use_clahe=config.use_clahe)
            if image is not None:
                images.append(image)
                labels.append(label)
    if len(images) == 0:
        return np.array([]), np.array([])
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)

def create_roi_cache(config):
    train_csvs = [config.calc_train_csv, config.mass_train_csv]
    train_images, train_labels = preprocess_roi_dataset(config, train_csvs, 'train')
    test_csvs = [config.calc_test_csv, config.mass_test_csv]
    test_images, test_labels = preprocess_roi_dataset(config, test_csvs, 'test')
    if len(train_images) > 0:
        val_ratio = config.val_split / (config.train_split + config.val_split)
        train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
            train_images, train_labels, test_size=val_ratio,
            random_state=RANDOM_SEED, stratify=train_labels
        )
        np.savez(config.cache_path / 'train_cache.npz', images=train_imgs, labels=train_lbls)
        np.savez(config.cache_path / 'val_cache.npz', images=val_imgs, labels=val_lbls)
        np.savez(config.cache_path / 'test_cache.npz', images=test_images, labels=test_labels)

## 4. Data Loading

In [ ]:
def remount_drive():
    if RUNTIME_ENV != "colab":
        return False
    try:
        from google.colab import drive
        print("Google Drive connection lost. Remounting...")
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        import time
        time.sleep(2)
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive remounted successfully")
        return True
    except Exception as e:
        print(f"Failed to remount: {e}")
        return False

def load_cached_data(config, max_retries=2):
    cache_files = {
        'train': config.cache_path / 'train_cache.npz',
        'val': config.cache_path / 'val_cache.npz',
        'test': config.cache_path / 'test_cache.npz'
    }
    if not all(f.exists() for f in cache_files.values()):
        print("Cache not found. Creating from DICOM files...")
        create_roi_cache(config)
    if not all(f.exists() for f in cache_files.values()):
        raise FileNotFoundError(f"Cache files not found in {config.cache_path}")
    train_images = train_labels = None
    val_images = val_labels = None
    test_images = test_labels = None
    for attempt in range(max_retries + 1):
        try:
            with np.load(cache_files['train']) as train_data:
                train_images = train_data['images'].copy()
                train_labels = train_data['labels'].copy()
            with np.load(cache_files['val']) as val_data:
                val_images = val_data['images'].copy()
                val_labels = val_data['labels'].copy()
            with np.load(cache_files['test']) as test_data:
                test_images = test_data['images'].copy()
                test_labels = test_data['labels'].copy()
            break
        except OSError as e:
            error_msg = str(e)
            if "Transport endpoint" in error_msg or "Input/output error" in error_msg:
                if attempt < max_retries:
                    print(f"Drive connection error (attempt {attempt + 1})")
                    gc.collect()
                    if remount_drive():
                        continue
                raise
            else:
                raise
    print(f"Loaded: Train={len(train_images)}, Val={len(val_images)}, Test={len(test_images)}")
    return (train_images, train_labels, val_images, val_labels, test_images, test_labels)

train_images, train_labels, val_images, val_labels, test_images, test_labels = load_cached_data(config)

## 5. Enhanced Data Pipeline

In [ ]:
@tf.function
def augment_images_v13(images):
    images = tf.image.random_flip_left_right(images)
    images = tf.image.random_flip_up_down(images)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    images = tf.image.rot90(images, k)
    images = tf.image.random_brightness(images, max_delta=30.0)
    images = tf.image.random_contrast(images, lower=0.85, upper=1.15)
    images = tf.image.random_saturation(images, lower=0.9, upper=1.1)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return images

@tf.function
def apply_efficientnet_preprocessing(images):
    return tf.keras.applications.efficientnet.preprocess_input(images)

def create_generator(images, labels, shuffle=False, seed=None):
    indices = np.arange(len(images))
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
    for idx in indices:
        yield images[idx], labels[idx]

def create_dataset(images, labels, config, training=True, shuffle_seed=None, use_reduced_batch=False):
    output_signature = (
        tf.TensorSpec(shape=(config.image_size[0], config.image_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )
    dataset = tf.data.Dataset.from_generator(
        lambda: create_generator(images, labels, shuffle=training, seed=shuffle_seed),
        output_signature=output_signature
    )
    if training:
        dataset = dataset.shuffle(config.shuffle_buffer)
    batch_size = config.batch_size_finetune if use_reduced_batch else config.batch_size
    dataset = dataset.batch(batch_size)
    if training:
        dataset = dataset.map(
            lambda x, y: (augment_images_v13(x), y),
            num_parallel_calls=config.num_parallel_calls
        )
    dataset = dataset.map(
        lambda x, y: (apply_efficientnet_preprocessing(x), y),
        num_parallel_calls=config.num_parallel_calls
    )
    dataset = dataset.prefetch(config.prefetch_buffer)
    return dataset

train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

## 6. Class Weight Computation

In [ ]:
def compute_class_weights(labels, malignant_multiplier=2.5):
    balanced = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=labels
    )
    weights = {
        0: balanced[0],
        1: balanced[1] * malignant_multiplier
    }
    return weights

config.class_weights = compute_class_weights(train_labels, config.malignant_weight_multiplier)
print(f"Class Weights: Benign={config.class_weights[0]:.3f}, Malignant={config.class_weights[1]:.3f}")

## 7. V13 Model Architecture

In [ ]:
def create_focal_loss(gamma=2.0, alpha=0.7):
    @tf.function
    def focal_loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = tf.pow(1 - p_t, gamma)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(alpha_t * focal_weight * ce)
    return focal_loss

def build_model_v13(config, seed=42):
    tf.random.set_seed(seed)
    np.random.seed(seed)
    inputs = layers.Input(shape=(config.image_size[0], config.image_size[1], 3))
    base_model = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        pooling=None
    )
    base_model.trainable = False
    x = base_model.output
    if config.use_spatial_dropout:
        x = layers.SpatialDropout2D(config.spatial_dropout_rate)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        config.dense_units,
        kernel_regularizer=regularizers.l2(config.l2_regularization),
        kernel_initializer='he_normal'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(config.dropout_rate)(x)
    x = layers.Dense(
        config.dense_units // 2,
        kernel_regularizer=regularizers.l2(config.l2_regularization),
        kernel_initializer='he_normal'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(config.dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    if config.use_focal_loss:
        loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
    else:
        loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
    model.compile(
        optimizer=Adam(learning_rate=config.stage1_lr, clipnorm=config.gradient_clip_norm),
        loss=loss,
        metrics=[
            BinaryAccuracy(name='accuracy'),
            AUC(name='auc'),
            Precision(name='precision'),
            Recall(name='recall')
        ]
    )
    return model

def unfreeze_layers(model, num_frozen_layers):
    for layer in model.layers[:num_frozen_layers]:
        layer.trainable = False
    for layer in model.layers[num_frozen_layers:]:
        layer.trainable = True

def get_trainable_summary(model):
    trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
    non_trainable = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
    return trainable, non_trainable

test_model = build_model_v13(config, seed=42)
trainable, non_trainable = get_trainable_summary(test_model)
print(f"Total parameters:     {trainable + non_trainable:,}")
print(f"Trainable:            {trainable:,}")
print(f"Non-trainable:        {non_trainable:,}")
print(f"Total layers:         {len(test_model.layers)}")
del test_model
gc.collect()

## 8. V13 Training Callbacks

In [ ]:
class WarmupLearningRateScheduler(Callback):
    def __init__(self, warmup_epochs, start_lr, target_lr):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.start_lr = start_lr
        self.target_lr = target_lr
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
            self.model.optimizer.learning_rate.assign(lr)

class CosineAnnealingScheduler(Callback):
    def __init__(self, max_epochs, initial_lr, min_lr=1e-7, warmup_epochs=0):
        super().__init__()
        self.max_epochs = max_epochs
        self.initial_lr = initial_lr
        self.min_lr = min_lr
        self.warmup_epochs = warmup_epochs
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            return
        adjusted_epoch = epoch - self.warmup_epochs
        adjusted_max = self.max_epochs - self.warmup_epochs
        cos_decay = 0.5 * (1 + np.cos(np.pi * adjusted_epoch / adjusted_max))
        lr = self.min_lr + (self.initial_lr - self.min_lr) * cos_decay
        self.model.optimizer.learning_rate.assign(lr)

class MinEpochEarlyStopping(Callback):
    def __init__(self, monitor='val_auc', patience=15, min_epochs=15, mode='max'):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_epochs = min_epochs
        self.mode = mode
        self.wait = 0
        self.best = -np.inf if mode == 'max' else np.inf
        self.stopped_epoch = 0
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        if epoch < self.min_epochs:
            self._update_best(current)
            return
        if self._is_improvement(current):
            self._update_best(current)
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.model.stop_training = True
                print(f"\nEarly stopping at epoch {epoch + 1}")
    def _is_improvement(self, current):
        if self.mode == 'max':
            return current > self.best
        return current < self.best
    def _update_best(self, current):
        self.best = current

def get_callbacks_v13(config, stage, model_idx, total_epochs):
    callbacks = []
    if stage == 1:
        target_lr = config.stage1_lr
        min_epochs = config.min_epochs_stage1
    elif stage == 2:
        target_lr = config.stage2_lr
        min_epochs = config.min_epochs_stage2
    else:
        target_lr = config.stage3_lr
        min_epochs = config.min_epochs_stage3
    if stage == 1 and config.use_lr_warmup:
        callbacks.append(WarmupLearningRateScheduler(
            warmup_epochs=config.warmup_epochs,
            start_lr=config.warmup_start_lr,
            target_lr=target_lr
        ))
    if config.use_cosine_annealing:
        warmup = config.warmup_epochs if stage == 1 else 0
        callbacks.append(CosineAnnealingScheduler(
            max_epochs=total_epochs,
            initial_lr=target_lr,
            min_lr=1e-7,
            warmup_epochs=warmup
        ))
    else:
        callbacks.append(ReduceLROnPlateau(
            monitor='val_auc',
            factor=config.lr_reduce_factor,
            patience=config.lr_reduce_patience,
            min_lr=1e-7,
            mode='max',
            verbose=1
        ))
    checkpoint_path = config.checkpoint_path / f'model_{model_idx}_stage{stage}_best.keras'
    callbacks.append(ModelCheckpoint(
        str(checkpoint_path),
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=0
    ))
    callbacks.append(MinEpochEarlyStopping(
        monitor='val_auc',
        patience=config.early_stop_patience,
        min_epochs=min_epochs,
        mode='max'
    ))
    return callbacks

## 9. V13 Training Functions

In [ ]:
def train_single_model_v13(config, train_ds, val_ds, train_steps, val_steps, model_idx=0):
    print(f"\n{'='*60}")
    print(f"TRAINING MODEL {model_idx + 1}/{config.ensemble_size} (V13 Optimized)")
    print(f"{'='*60}")
    seed = config.ensemble_seeds[model_idx]
    model = build_model_v13(config, seed=seed)
    combined_history = {
        'loss': [], 'val_loss': [],
        'accuracy': [], 'val_accuracy': [],
        'auc': [], 'val_auc': [],
        'precision': [], 'val_precision': [],
        'recall': [], 'val_recall': [],
        'lr': []
    }
    
    # STAGE 1
    print(f"\n--- Stage 1: Head only ({config.stage1_epochs} epochs, LR={config.stage1_lr}) ---")
    print_memory_usage("Stage 1 Start")
    callbacks = get_callbacks_v13(config, stage=1, model_idx=model_idx, total_epochs=config.stage1_epochs)
    history1 = model.fit(
        train_ds, epochs=config.stage1_epochs, steps_per_epoch=train_steps,
        validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=callbacks, verbose=1
    )
    for key in combined_history:
        if key in history1.history:
            combined_history[key].extend(history1.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history1.history['loss']))
    gc.collect()
    
    # STAGE 2 (KEY STAGE)
    print(f"\n--- Stage 2: Partial unfreeze ({config.stage2_epochs} epochs, LR={config.stage2_lr}) ---")
    print(f"    Freezing first {config.stage2_frozen_layers} layers (conservative)")
    print_memory_usage("Stage 2 Start")
    unfreeze_layers(model, config.stage2_frozen_layers)
    if config.use_focal_loss:
        loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
    else:
        loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
    model.compile(
        optimizer=Adam(learning_rate=config.stage2_lr, clipnorm=config.gradient_clip_norm),
        loss=loss,
        metrics=[BinaryAccuracy(name='accuracy'), AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
    )
    callbacks = get_callbacks_v13(config, stage=2, model_idx=model_idx, total_epochs=config.stage2_epochs)
    history2 = model.fit(
        train_ds, epochs=config.stage2_epochs, steps_per_epoch=train_steps,
        validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=callbacks, verbose=1
    )
    for key in combined_history:
        if key in history2.history:
            combined_history[key].extend(history2.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history2.history['loss']))
    stage2_best = config.checkpoint_path / f'model_{model_idx}_stage2_best.keras'
    print(f"\n Stage 2 best saved: {stage2_best.name}")
    gc.collect()
    
    # STAGE 3 (CONSERVATIVE)
    print(f"\n--- Stage 3: Conservative fine-tune ({config.stage3_epochs} epochs, LR={config.stage3_lr}) ---")
    print(f"    Keeping first {config.stage3_frozen_layers} layers frozen")
    print_memory_usage("Stage 3 Start")
    unfreeze_layers(model, config.stage3_frozen_layers)
    model.compile(
        optimizer=Adam(learning_rate=config.stage3_lr, clipnorm=config.gradient_clip_norm),
        loss=loss,
        metrics=[BinaryAccuracy(name='accuracy'), AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
    )
    callbacks = get_callbacks_v13(config, stage=3, model_idx=model_idx, total_epochs=config.stage3_epochs)
    history3 = model.fit(
        train_ds, epochs=config.stage3_epochs, steps_per_epoch=train_steps,
        validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=callbacks, verbose=1
    )
    for key in combined_history:
        if key in history3.history:
            combined_history[key].extend(history3.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history3.history['loss']))
    
    # Load best checkpoint
    stage2_best = config.checkpoint_path / f'model_{model_idx}_stage2_best.keras'
    stage3_best = config.checkpoint_path / f'model_{model_idx}_stage3_best.keras'
    s2_best_auc = max(history2.history['val_auc']) if history2.history['val_auc'] else 0
    s3_best_auc = max(history3.history['val_auc']) if history3.history['val_auc'] else 0
    if s2_best_auc >= s3_best_auc and stage2_best.exists():
        print(f"\n Stage 2 was better ({s2_best_auc:.4f} vs {s3_best_auc:.4f})")
        model.load_weights(str(stage2_best))
    elif stage3_best.exists():
        print(f"\n Stage 3 was better ({s3_best_auc:.4f} vs {s2_best_auc:.4f})")
        model.load_weights(str(stage3_best))
    print_memory_usage("Training Complete")
    print(f"\n Model {model_idx + 1} training complete!")
    return model, combined_history

def train_ensemble_v13(config, train_images, train_labels, val_images, val_labels):
    model_paths = []
    histories = []
    train_steps = len(train_labels) // config.batch_size
    val_steps = len(val_labels) // config.batch_size
    print(f"\n{'='*60}")
    print(f"V13 ENSEMBLE TRAINING: {config.ensemble_size} model(s)")
    print(f"Training samples: {len(train_labels)}")
    print(f"Validation samples: {len(val_labels)}")
    print(f"Batch size: {config.batch_size}")
    print(f"{'='*60}")
    for model_idx in range(config.ensemble_size):
        seed = config.ensemble_seeds[model_idx]
        train_ds = create_dataset(train_images, train_labels, config, training=True, shuffle_seed=seed)
        val_ds = create_dataset(val_images, val_labels, config, training=False)
        model, history = train_single_model_v13(config, train_ds, val_ds, train_steps, val_steps, model_idx)
        model_save_path = config.checkpoint_path / f'model_{model_idx}_final.keras'
        model.save(str(model_save_path))
        model_paths.append(model_save_path)
        histories.append(history)
        print(f"Model {model_idx + 1} saved to {model_save_path.name}")
        del model
        del train_ds
        del val_ds
        tf.keras.backend.clear_session()
        gc.collect()
        print_memory_usage(f"After Model {model_idx + 1}")
    print(f"\n{'='*60}")
    print("RELOADING MODELS FOR ENSEMBLE")
    print(f"{'='*60}")
    models_list = []
    for path in model_paths:
        print(f"Loading {path.name}...")
        model = tf.keras.models.load_model(str(path), compile=False)
        models_list.append(model)
    print(f"\n Ensemble of {len(models_list)} model(s) ready!")
    return models_list, histories

## 10. Execute Training

In [ ]:
total_epochs = (config.stage1_epochs + config.stage2_epochs + config.stage3_epochs) * config.ensemble_size
print("-" * 60)
print("V13 TRAINING CONFIGURATION")
print("-" * 60)
print(f"Ensemble Models:    {config.ensemble_size}")
print(f"Training Samples:   {len(train_labels)}")
print(f"Validation Samples: {len(val_labels)}")
print(f"Total Epochs:       {total_epochs}")
print(f"Regularization:     L2={config.l2_regularization}, Dropout={config.dropout_rate}")
print("-" * 60)

ensemble_models, training_histories = train_ensemble_v13(
    config, train_images, train_labels, val_images, val_labels
)

## 11. Test-Time Augmentation

In [ ]:
def apply_tta_single(image):
    yield image
    yield np.flip(image, axis=1)
    yield np.flip(image, axis=0)
    yield np.flip(np.flip(image, axis=1), axis=0)
    yield np.rot90(image, k=1)
    yield np.rot90(image, k=2)
    yield np.rot90(image, k=3)
    yield np.flip(np.rot90(image, k=1), axis=1)

def predict_with_tta(models, images, config):
    n_samples = len(images)
    n_models = len(models)
    n_tta = config.tta_augmentations if config.use_tta else 1
    all_predictions = np.zeros(n_samples, dtype=np.float32)
    tta_batch_size = min(config.batch_size_inference, 64)
    if not config.use_tta:
        prediction_count = 0
        for model in models:
            for start_idx in range(0, n_samples, tta_batch_size):
                end_idx = min(start_idx + tta_batch_size, n_samples)
                batch = images[start_idx:end_idx].copy()
                batch_preprocessed = efficientnet_preprocess(batch.astype(np.float32))
                preds = model.predict(batch_preprocessed, verbose=0, batch_size=tta_batch_size)
                preds_flat = np.nan_to_num(preds.flatten(), nan=0.5)
                all_predictions[start_idx:end_idx] += preds_flat
                del batch, batch_preprocessed, preds
            prediction_count += 1
            gc.collect()
        return np.nan_to_num(all_predictions / prediction_count, nan=0.5)
    print(f"Running TTA with {n_tta} augmentations on {n_samples} images ({n_models} models)...")
    nan_count = 0
    for img_idx in tqdm(range(n_samples), desc="TTA Prediction"):
        img = images[img_idx]
        img_predictions = []
        tta_count = 0
        for aug_img in apply_tta_single(img):
            if tta_count >= n_tta:
                break
            aug_batch = np.expand_dims(aug_img, axis=0).astype(np.float32)
            aug_preprocessed = efficientnet_preprocess(aug_batch)
            for model in models:
                pred = model.predict(aug_preprocessed, verbose=0, batch_size=1)
                pred_val = float(pred[0, 0])
                if np.isnan(pred_val) or np.isinf(pred_val):
                    nan_count += 1
                    pred_val = 0.5
                img_predictions.append(pred_val)
            del aug_batch, aug_preprocessed
            tta_count += 1
        if img_predictions:
            mean_pred = np.nanmean(img_predictions)
            all_predictions[img_idx] = mean_pred if not np.isnan(mean_pred) else 0.5
        else:
            all_predictions[img_idx] = 0.5
        if img_idx % 100 == 0:
            gc.collect()
    if nan_count > 0:
        print(f"Warning: {nan_count} NaN predictions replaced with 0.5")
    gc.collect()
    return np.nan_to_num(all_predictions, nan=0.5)

## 12. Evaluation

In [ ]:
def evaluate_ensemble(models, test_images, test_labels, config):
    predictions = predict_with_tta(models, test_images, config)
    predictions = np.nan_to_num(predictions, nan=0.5, posinf=1.0, neginf=0.0)
    predictions = np.clip(predictions, 0.0, 1.0)
    predicted_classes = (predictions > 0.5).astype(int)
    auc_score = roc_auc_score(test_labels, predictions)
    accuracy = accuracy_score(test_labels, predicted_classes)
    precision = precision_score(test_labels, predicted_classes, zero_division=0)
    recall = recall_score(test_labels, predicted_classes, zero_division=0)
    f1 = f1_score(test_labels, predicted_classes, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(test_labels, predicted_classes).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr, tpr, thresholds = roc_curve(test_labels, predictions)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    optimal_predicted = (predictions > optimal_threshold).astype(int)
    optimal_accuracy = accuracy_score(test_labels, optimal_predicted)
    optimal_recall = recall_score(test_labels, optimal_predicted, zero_division=0)
    tn_opt, fp_opt, fn_opt, tp_opt = confusion_matrix(test_labels, optimal_predicted).ravel()
    optimal_specificity = tn_opt / (tn_opt + fp_opt) if (tn_opt + fp_opt) > 0 else 0
    results = {
        'auc': float(auc_score),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'sensitivity': float(recall),
        'specificity': float(specificity),
        'f1_score': float(f1),
        'optimal_threshold': float(optimal_threshold),
        'optimal_accuracy': float(optimal_accuracy),
        'optimal_sensitivity': float(optimal_recall),
        'optimal_specificity': float(optimal_specificity),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'predictions': predictions.tolist(),
        'true_labels': test_labels.tolist(),
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist()
    }
    return results

results = evaluate_ensemble(ensemble_models, test_images, test_labels, config)
print("=" * 60)
print("V13 OPTIMIZED EVALUATION RESULTS")
print("=" * 60)
print()
print("Metrics (threshold=0.5):")
print(f"  AUC:           {results['auc']:.4f}")
print(f"  Accuracy:      {results['accuracy']:.4f}")
print(f"  Sensitivity:   {results['sensitivity']:.4f}")
print(f"  Specificity:   {results['specificity']:.4f}")
print(f"  F1 Score:      {results['f1_score']:.4f}")
print()
print(f"Optimal Threshold ({results['optimal_threshold']:.3f}):")
print(f"  Accuracy:      {results['optimal_accuracy']:.4f}")
print(f"  Sensitivity:   {results['optimal_sensitivity']:.4f}")
print(f"  Specificity:   {results['optimal_specificity']:.4f}")
print()
print("Confusion Matrix:")
cm = results['confusion_matrix']
print(f"  TN={cm['tn']}, FP={cm['fp']}")
print(f"  FN={cm['fn']}, TP={cm['tp']}")
print()
print("=" * 60)
print("COMPARISON WITH V12:")
print(f"  V12 AUC:   0.7700")
print(f"  V13 AUC:   {results['auc']:.4f}")
print(f"  Delta:     {results['auc'] - 0.7700:+.4f}")
print("=" * 60)

## 13. Visualization

In [ ]:
def plot_roc_curve(results, save_path=None):
    plt.figure(figsize=(8, 8))
    plt.plot(results['fpr'], results['tpr'], color='blue', lw=2, label=f"V13 ROC (AUC = {results['auc']:.4f})")
    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - EfficientNet-B4 V13 Optimized', fontsize=14)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(True, alpha=0.3)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

def plot_training_history(histories, save_path=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, history in enumerate(histories):
        label_suffix = f" (Model {idx})"
        axes[0, 0].plot(history['loss'], label=f'Train{label_suffix}', alpha=0.7)
        axes[0, 0].plot(history['val_loss'], label=f'Val{label_suffix}', linestyle='--', alpha=0.7)
        axes[0, 1].plot(history['auc'], label=f'Train{label_suffix}', alpha=0.7)
        axes[0, 1].plot(history['val_auc'], label=f'Val{label_suffix}', linestyle='--', alpha=0.7)
        axes[1, 0].plot(history['accuracy'], label=f'Train{label_suffix}', alpha=0.7)
        axes[1, 0].plot(history['val_accuracy'], label=f'Val{label_suffix}', linestyle='--', alpha=0.7)
        if 'lr' in history:
            axes[1, 1].plot(history['lr'], label=f'LR{label_suffix}', alpha=0.7)
    axes[0, 0].set_title('Loss', fontsize=12)
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].set_title('AUC', fontsize=12)
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3)
    axes[1, 0].set_title('Accuracy', fontsize=12)
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].set_title('Learning Rate', fontsize=12)
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_roc_curve(results, save_path=config.results_path / 'v13_roc_curve.png')
plot_training_history(training_histories, save_path=config.results_path / 'v13_training_history.png')

## 14. Save Results

In [ ]:
for idx, model in enumerate(ensemble_models):
    model_path = config.results_path / f'v13_model_{idx}.keras'
    model.save(str(model_path))

results_with_config = {
    'experiment': 'EfficientNetB4_V13_Optimized',
    'architecture': 'EfficientNet-B4',
    'version': 'V13',
    'improvements': [
        'Increased L2 regularization (3e-4)',
        'Higher dropout (0.5)',
        'Spatial dropout (0.2)',
        'Two-layer classification head',
        'Conservative unfreezing',
        'Reduced Stage 3 (25 epochs)',
        'Cosine annealing LR',
        'Enhanced augmentation'
    ],
    'configuration': {
        'image_size': list(config.image_size),
        'batch_size': config.batch_size,
        'ensemble_size': config.ensemble_size,
        'stage1_epochs': config.stage1_epochs,
        'stage2_epochs': config.stage2_epochs,
        'stage3_epochs': config.stage3_epochs,
        'dropout_rate': config.dropout_rate,
        'l2_regularization': config.l2_regularization,
        'spatial_dropout': config.spatial_dropout_rate,
        'label_smoothing': config.label_smoothing,
        'stage2_frozen_layers': config.stage2_frozen_layers,
        'stage3_frozen_layers': config.stage3_frozen_layers
    },
    'results': {
        'auc': results['auc'],
        'accuracy': results['accuracy'],
        'sensitivity': results['sensitivity'],
        'specificity': results['specificity'],
        'precision': results['precision'],
        'f1_score': results['f1_score'],
        'optimal_threshold': results['optimal_threshold'],
        'optimal_accuracy': results['optimal_accuracy'],
        'optimal_sensitivity': results['optimal_sensitivity'],
        'optimal_specificity': results['optimal_specificity']
    },
    'comparison': {
        'v12_auc': 0.7700,
        'v13_auc': results['auc'],
        'improvement': results['auc'] - 0.7700
    },
    'confusion_matrix': results['confusion_matrix']
}

with open(config.results_path / 'v13_results.json', 'w') as f:
    json.dump(results_with_config, f, indent=2)

for idx, history in enumerate(training_histories):
    history_path = config.results_path / f'v13_history_{idx}.json'
    serializable_history = {k: [float(v) for v in vals] for k, vals in history.items()}
    with open(history_path, 'w') as f:
        json.dump(serializable_history, f, indent=2)

print(f"Results saved to: {config.results_path}")

## 15. Summary

In [ ]:
print("=" * 70)
print("V13 OPTIMIZED EXPERIMENT SUMMARY")
print("=" * 70)
print()
print("Key Improvements from V12:")
print(f"  L2 Regularization:  1e-4 -> {config.l2_regularization}")
print(f"  Dropout:            0.4 -> {config.dropout_rate}")
print(f"  Spatial Dropout:    NEW -> {config.spatial_dropout_rate}")
print(f"  Stage 2 frozen:     300 -> {config.stage2_frozen_layers}")
print(f"  Stage 3 frozen:     0 -> {config.stage3_frozen_layers}")
print(f"  Stage 3 epochs:     80 -> {config.stage3_epochs}")
print(f"  Early stop:         25 -> {config.early_stop_patience}")
print()
print("Results:")
print(f"  AUC:                {results['auc']:.4f}")
print(f"  Accuracy:           {results['accuracy']:.4f}")
print(f"  Sensitivity:        {results['sensitivity']:.4f}")
print(f"  Specificity:        {results['specificity']:.4f}")
print(f"  F1 Score:           {results['f1_score']:.4f}")
print()
print("Comparison with V12:")
print(f"  V12 AUC:            0.7700")
print(f"  V13 AUC:            {results['auc']:.4f}")
print(f"  Delta:              {results['auc'] - 0.7700:+.4f}")
print()
print("=" * 70)
print(f"Results saved to: {config.results_path}")
print("=" * 70)